# HubbardBath 01 — Fermi–Hubbard foundations

This notebook builds the **two-site spinful Fermi–Hubbard model from first principles**. We explicitly construct the fermionic operators, verify their algebra, build the Hamiltonian, isolate the half-filled sector, and inspect how the spectrum changes with interaction strength.

**Milestone question:** Can we construct a correct two-site Fermi–Hubbard Hamiltonian and validate the algebra before adding temperature, dissipation, or quantum algorithms?


## 1. Physical model

Mode ordering:

```text
0 = site 0 ↑
1 = site 0 ↓
2 = site 1 ↑
3 = site 1 ↓
```

The Hamiltonian is

$$H=-t\sum_\sigma(c^\dagger_{0\sigma}c_{1\sigma}+c^\dagger_{1\sigma}c_{0\sigma})+U\sum_i n_{i\uparrow}n_{i\downarrow}-\mu\sum_{i,\sigma}n_{i\sigma}.$$

`t` controls hopping, `U` penalizes double occupancy, and `mu` is the chemical potential. We use `t=1` as the energy scale.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'HubbardBath']
for candidate in candidates:
    if (candidate / 'src' / 'hubbard.py').exists():
        repo_root = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run this notebook from the HubbardBath repository.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.hubbard import (
    annihilation_operator, anticommutator, commutator,
    fock_label, project_to_particle_sector, sector_indices,
    total_number_operator, two_site_hubbard_hamiltonian,
)

np.set_printoptions(precision=4, suppress=True)
print('Repository:', repo_root)


## 2. Fermionic anti-commutation

Fermions satisfy
$$\{c_i,c_j^\dagger\}=\delta_{ij}I,\qquad \{c_i,c_j\}=0.$$
The Jordan–Wigner parity string enforces the exchange signs. We test the relations numerically before trusting any later result.


In [ ]:
num_modes = 4
dim = 2 ** num_modes
identity = np.eye(dim, dtype=complex)
zero = np.zeros((dim, dim), dtype=complex)

max_car_error = 0.0
for i in range(num_modes):
    ci = annihilation_operator(num_modes, i)
    for j in range(num_modes):
        cj = annihilation_operator(num_modes, j)
        expected = identity if i == j else zero
        e1 = np.linalg.norm(anticommutator(ci, cj.conj().T) - expected)
        e2 = np.linalg.norm(anticommutator(ci, cj))
        max_car_error = max(max_car_error, e1, e2)

print(f'Maximum anti-commutation error: {max_car_error:.3e}')
assert max_car_error < 1e-12


## 3. Build and validate the Hamiltonian

Four fermionic modes give a full Fock-space dimension of $2^4=16$. We verify Hermiticity and conservation of total particle number.


In [ ]:
t, U, mu = 1.0, 4.0, 0.0
H = two_site_hubbard_hamiltonian(t=t, U=U, mu=mu)
N = total_number_operator(4)

hermiticity_error = np.linalg.norm(H - H.conj().T)
number_conservation_error = np.linalg.norm(commutator(H, N))

print('Hamiltonian shape:', H.shape)
print(f'Hermiticity error: {hermiticity_error:.3e}')
print(f'Particle-number conservation error: {number_conservation_error:.3e}')
assert hermiticity_error < 1e-12
assert number_conservation_error < 1e-12


## 4. Half filling

With two lattice sites, half filling means two electrons. The $N=2$ sector has only $\binom{4}{2}=6$ basis states, so we can work in the physically relevant block exactly.


In [ ]:
half_indices = sector_indices(num_modes=4, particle_number=2)
H_half, _ = project_to_particle_sector(H, num_modes=4, particle_number=2)

print('Half-filled basis states:')
for index in half_indices:
    print(f'{index:2d}  {fock_label(index, 4)}')
print('\nHalf-filled Hamiltonian shape:', H_half.shape)
assert H_half.shape == (6, 6)


## 5. Exact spectrum

At `U/t = 4`, diagonalize the six-dimensional half-filled block exactly.


In [ ]:
energies = np.linalg.eigvalsh(H_half)
print('Half-filled energies for U/t = 4:')
print(energies)


## 6. First parameter experiment: vary interaction strength

The ratio $U/t$ controls the competition between electron motion and on-site repulsion. We sweep it and track the exact energy levels in the half-filled sector.


In [ ]:
U_values = np.linspace(0.0, 8.0, 81)
all_energies = []

for U_value in U_values:
    H_u = two_site_hubbard_hamiltonian(t=1.0, U=U_value, mu=0.0)
    H_u_half, _ = project_to_particle_sector(H_u, num_modes=4, particle_number=2)
    all_energies.append(np.linalg.eigvalsh(H_u_half))

all_energies = np.asarray(all_energies)
plt.figure(figsize=(7, 4.5))
for level in range(all_energies.shape[1]):
    plt.plot(U_values, all_energies[:, level], linewidth=1.6)
plt.xlabel(r'Interaction strength $U/t$')
plt.ylabel(r'Energy $E/t$')
plt.title('Two-site Fermi–Hubbard spectrum at half filling')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## What this milestone establishes

1. The fermionic operators satisfy the canonical anti-commutation relations.
2. The two-site Hubbard Hamiltonian is Hermitian.
3. Total particle number is conserved.
4. The half-filled problem is exactly represented by a six-dimensional sector.

**Next milestone:** construct the Gibbs state $\rho_\beta=e^{-\beta H}/\mathrm{Tr}(e^{-\beta H})$ and study energy, entropy, double occupancy, and spin correlations as functions of `U/t` and temperature.
